# ML Masterclass 03: Tree-Based Models & Ensembles

Welcome to Notebook 03. For tabular structured data (CSVs, SQL databases), Tree Ensembles—specifically Gradient Boosting Machines like XGBoost and LightGBM—are the undisputed champions. They consistently outperform Deep Neural Networks on these tasks.

In this notebook, we strip away the magic of `RandomForestClassifier()` and build the core logic from scratch using Information Theory.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"Why does a Random Forest almost never overfit as you add more trees (e.g., going from 100 to 1000 trees), but a Gradient Boosting Machine will inevitably overfit and memorize the noise if you add too many trees?"*

---

## Table of Contents
1. **Information Theory:** Entropy, Gini Impurity, and Information Gain.
2. **The Base Learner:** Writing a recursive binary splitting Decision Tree from scratch.
3. **Bagging (Random Forests):** Reducing Variance through Bootstrap Aggregation.
4. **Boosting (GBMs):** Reducing Bias by mathematically fitting to pseudo-residuals.


## 1. Information Theory & Node Splitting

A decision tree is a greedy algorithm. At every step, it asks: *"What single Yes/No question about a feature will divide my data into the most purely separated groups?"*

How do we mathematically measure "purity"? 

*   **Shannon Entropy:** Measures the amount of chaos or surprise in a system. $H = -\sum p(x) \log_2 p(x)$. If a node is 50/50 Cats and Dogs, Entropy is maxed out at 1.0 (total chaos). If a node is 100% Cats, Entropy is 0.0 (perfect purity).
*   **Gini Impurity:** Measures the probability of mislabeling a randomly chosen element. $G = 1 - \sum p(x)^2$. Computationally slightly faster than Entropy because it avoids the $\log$ calculation.

**Information Gain** is simply the Entropy of the parent node MINUS the weighted average Entropy of the child nodes. The tree picks the split with the highest Information Gain.

In [ ]:
import numpy as np
from collections import Counter

# -----------------------------------------------------
# IMPLEMENTATION: Entropy and Information Gain
# -----------------------------------------------------

def calculate_entropy(y):
    """Calculates Shannon Entropy of a label array."""
    counts = np.bincount(y)
    probabilities = counts / len(y)
    
    entropy = 0
    for p in probabilities:
        if p > 0:
            entropy -= p * np.log2(p)
    return entropy

def calculate_information_gain(y, y_left, y_right):
    """Calculates IG based on parent and split children."""
    # Parent entropy
    parent_entropy = calculate_entropy(y)
    
    # Weighted average of children entropy
    n = len(y)
    n_l, n_r = len(y_left), len(y_right)
    
    if n_l == 0 or n_r == 0:
        return 0
        
    child_entropy = (n_l / n) * calculate_entropy(y_left) + (n_r / n) * calculate_entropy(y_right)
    
    # Information Gain
    return parent_entropy - child_entropy

# Scenario: Parent node has 5 Cats (0) and 5 Dogs (1)
y_parent = np.array([0,0,0,0,0, 1,1,1,1,1])

# Split A is terrible (Still mixed 50/50 in both branches)
y_left_A = np.array([0,0,0, 1,1])
y_right_A = np.array([0,0, 1,1,1])

# Split B is perfect (100% pure)
y_left_B = np.array([0,0,0,0,0])
y_right_B = np.array([1,1,1,1,1])

print(f"Parent Entropy (Chaos level): {calculate_entropy(y_parent):.2f}")
print(f"Information Gain for Split A (Terrible): {calculate_information_gain(y_parent, y_left_A, y_right_A):.2f}")
print(f"Information Gain for Split B (Perfect): {calculate_information_gain(y_parent, y_left_B, y_right_B):.2f}")

## 2. Bagging & Boosting (The Ensemble Theory)

A single unrestricted Decision Tree will perfectly memorize its training data (100% accuracy, Massive Variance, extreme overfitting).

### BAGGING (Random Forests) -> Reduces Variance
*   **How it works:** Build 1,000 independent deep decision trees. Each tree only sees a random sub-sample of the data (bootstrapping) AND a random sub-sample of the features at each split.
*   **Why it works:** Because the trees are independent and randomized, their errors are uncorrelated. Averaging them cancels out the variance while preserving the predictive signal.

### BOOSTING (XGBoost) -> Reduces Bias
*   **How it works:** Build 1,000 highly restricted, shallow decision trees (stumps) **sequentially**.
*   **The Math:** Tree 1 predicts $Y$. It has some error (residuals). Tree 2 does NOT try to predict $Y$. Tree 2 deliberately tries to predict the *residuals* of Tree 1. Tree 3 predicts the residuals of Tree 2. 
*   **Why it works:** It acts exactly like Gradient Descent in function space, slowly stepping toward the true answer by fixing the mistakes of the previous steps.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does a Random Forest rarely overfit as you add trees, but a GBM will?*

**A:** In a Random Forest, trees are built independently in parallel. Adding more trees is just taking a larger, more stable average of a randomized process. By the Law of Large Numbers, the average simply stabilizes. 
In Gradient Boosting, trees are built sequentially to explicitly correct the errors of all previous trees. If you add 1,000 trees, eventually it will perfectly model the true signal, run out of signal, and start using the subsequent trees to explicitly memorize and model the random noise in your training set (overfitting).

### 🎓 DEEP QUESTION 2 ANSWERED
**Q:** *Decision Trees cannot extrapolate. If your training data target ranges from 0 to 100, what will a tree predict for a new sample larger than anything it's seen?*

**A:** It will predict exactly the average value of the highest leaf node it has (which will be $\le 100$). Trees are essentially giant IF/ELSE statements that chop the geometric space into boxes. If a point falls into the "maximum" box out on the edge of the known universe, the tree just returns the pre-calculated average of the training data in that box. Unlike Linear Regression ($y = mx + b$), trees have no concept of a line that extends to infinity.